In [1]:
from google import genai
from google.genai import types
import base64
import asyncio

In [44]:
import time, json
import re, logging

## Prompt Experiment

### exp1

In [120]:
text_prompt = """You are an expert video analyst. Given a video clip between 10 and 180 seconds in duration, please identify and return a continuous time range (at least 5 seconds long) that meets all of the following conditions. The identified range must be continuous and have no edits or cuts within it. Explain why the segment is selected and provide the specific scores.

    Conditions:

    Condition 1: Single Person Focus: The entire frame should primarily contain only one identifiable human being. There should be no other person's face clearly visible in the frame or in a position to distract. There should be no camera cuts to any other individuals.

    Condition 2: Immobility: The person must be either sitting or standing completely still. They should exhibit no significant movements, including:
    - Walking or running.
    - Shifting weight noticeably.
    - Excessive gesturing with hands or arms that obstruct the face.
    - Significant head nods or shakes.
    - Any other body language that would be considered more than minor fidgeting.

    Condition 3: Unobstructed Mouth: When the person is speaking, their mouth must be completely unobstructed. This means:
    - No part of any object (e.g., microphone, hands, clothing) should be in front of the person's mouth during speaking.
    - Facial hair that obscures the mouth is acceptable as long as the mouth's movement can be seen.
    - The person must be facing the camera relatively directly so that the mouth is clearly visible.

    Condition 4: Consistent Individual: The individual in the frame must be the same throughout the entire selected time range.

    Condition 5: Static Background: The background must be clear of any moving objects or people. Minor, infrequent movements in the background (e.g., a curtain swaying in a gentle breeze) are acceptable. However, any significant or frequent movements (e.g., people walking by, vehicles passing) are unacceptable. Gradual changes in background lighting (e.g., clouds passing in front of the sun) are acceptable, but sudden changes in lighting (e.g., flickering lights) are not.

    Condition 6: No Edits: The identified video segment must be continuous and has not been edited or cut within the range.

    Condition 7: Consistent Lighting: The lighting must be stable, with no noticeable flickering or changes in brightness. The stability of the lighting can be assessed based on the variation in brightness within the frame.

    Scoring Criteria:

    After identifying all time ranges that meet the above conditions, score each range based on the following criteria (higher score is better):

    Criteria 1: Face Angle (0-10 points):
    - 10 points: Person is facing the camera directly. Eyes and nose are pointed directly at the camera.
    - 5 points: Face is turned approximately 45 degrees to the side.
    - 0 points: Person is facing completely away from the camera. The face is not visible at all.
    - Linear interpolation for angles in between. Estimate the angle based on the relative position of facial features (e.g., eyes, nose, mouth) to the camera.

    Criteria 2: Background Cleanliness (0-10 points):
    - 10 points: Background is completely static and uncluttered.
    - 0 points: Background is very cluttered and distracting.
    - Subjective assessment based on the level of distraction.

    Criteria 3: Person Size (0-10 points):
    - 10 points: Person occupies a large portion of the frame. The person occupies 80% or more of the vertical height of the frame.
    - 0 points: Person is very small in the frame. The person occupies 20% or less of the vertical height of the frame.
    - Linear interpolation based on the percentage of the frame occupied by the person. Calculate the percentage using the vertical distance from the person's head to their feet, divided by the vertical height of the frame.

    Criteria 4: Person Movement (0-10 points):
    - 10 points: Person is completely still.
    - 0 points: Person is exhibiting significant movement.
    - Subjective assessment based on the amount and intensity of movement.

    Criteria 5: Face Lighting (0-10 points):
    - 10 points: High light intensity, facial features are clearly visible.
    - 5 points: Medium light intensity, some facial features are partially obscured by shadows.
    - 0 points: Low light intensity, facial features are barely visible.

    Selection Process:

    Step 1: Identify all continuous time ranges (at least 5 seconds long) that meet all conditions 1-7.

    Step 2: For each identified time range, calculate a total score by summing the scores for Face Angle, Background Cleanliness, Person Size, Person Movement, and Face Lighting.

    Step 3: Select the time range with the highest total score.

    Step 4: Provide the start and end times (in seconds) for the selected range.

    Step 5: Provide the individual scores for Face Angle, Background Cleanliness, Person Size, and Person Movement, as well as the total score for the selected range.

    Step 6: Explain why the segment is selected based on the scoring criteria.

    If no range meets all of these conditions, state \"No suitable range found.\"

    Your response should be a JSON string with the following structure:
    {
      "start_time": [Start Time in seconds],
      "end_time": [End Time in seconds],
      "face_angle_score": [Score],
      "background_cleanliness_score": [Score],
      "person_size_score": [Score],
      "person_movement_score": [Score],
      "face_lighting_score": [Score],
      "total_score": [Total Score],
      "suitable_range_found": [true/false]
    }
    
    Example Output (with a suitable range found):
    {
      "start_time": 15,
      "end_time": 20,
      "face_angle_score": 8,
      "background_cleanliness_score": 8,
      "person_size_score": 7,
      "person_movement_score": 10,
      "face_lighting_score": 8,
      "total_score": 41,
      "suitable_range_found": true
    }
    
    Example Output (if no suitable range is found):
    {
      "start_time": null,
      "end_time": null,
      "face_angle_score": null,
      "background_cleanliness_score": null,
      "person_size_score": null,
      "person_movement_score": null,
      "face_lighting_score": null,
      "total_score": null,
      "suitable_range_found": false
    }
    
    Return ONLY the JSON string as specified in the instructions. Do not include any additional text, explanations, or justifications.
    Let's start to analyze the video following the steps outlined above step by step."""

### exp2

In [9]:
text_prompt = """
Objective:

Identify a 5-second time range within the provided video clip that best meets the following conditions, and provide a score for each condition, as well as a total score.

Conditions:

Speaking: The person must be speaking clearly and audibly.
Immobility: The person must be either sitting or standing completely still. Any visible movement, including but not limited to head nods, shakes, fidgeting, or shifting weight, is unacceptable. The person's posture should remain virtually unchanged throughout the selected range.
Unobstructed Mouth: When the person is speaking, their mouth must be 100% unobstructed. No part of any object (including hands, clothing, or microphones) can be in front of the mouth at any time during speech. The mouth must be clearly and fully visible.
Appropriate Face Angle: The person's face should be angled towards the camera, allowing for a clear view of their features.
Good Face Lighting: The person's face should be well-lit and easily visible.
Clean Background: The background should be relatively uncluttered and free of distractions.
Consistent Lighting: The lighting must be stable, with no noticeable flickering or abrupt changes in brightness. Gradual changes in background lighting (e.g., clouds passing in front of the sun) are acceptable, but any sudden or significant change in the overall brightness of the person's face is unacceptable.
Appropriate Person Size: The person's face should occupy a reasonable portion of the frame, neither too small nor too large.
Scoring System (for each 5-second range):

For each condition, assign a score from 0 to 10, where:

10 = Perfectly meets the condition.
5 = Partially meets the condition.
0 = Does not meet the condition at all.
Selection Process:

Review the entire video clip.
Identify all potential 5-second time ranges where the person is speaking.
For each identified time range, calculate a total score by summing the scores for Face Angle, Background Cleanliness, Person Size, Person Movement, and Face Lighting. Before scoring, thoroughly verify each potential range, frame-by-frame, to ensure it absolutely meets all conditions. Discard any range that fails to meet even a single condition.
Select the 5-second time range with the highest total score. If no range meets all conditions, indicate "No suitable range found.

Your response should be a JSON string with the following structure:
{
  "start_time": [Start Time in seconds],
  "end_time": [End Time in seconds],
  "face_angle_score": [Score],
  "background_cleanliness_score": [Score],
  "person_size_score": [Score],
  "person_movement_score": [Score],
  "face_lighting_score": [Score],
  "total_score": [Total Score],
  "suitable_range_found": [true/false]
}

Example Output (with a suitable range found):
{
  "start_time": 15,
  "end_time": 20,
  "face_angle_score": 8,
  "background_cleanliness_score": 8,
  "person_size_score": 7,
  "person_movement_score": 10,
  "face_lighting_score": 8,
  "total_score": 41,
  "suitable_range_found": true
}

Example Output (if no suitable range is found):
{
  "start_time": null,
  "end_time": null,
  "face_angle_score": null,
  "background_cleanliness_score": null,
  "person_size_score": null,
  "person_movement_score": null,
  "face_lighting_score": null,
  "total_score": null,
  "suitable_range_found": false
}

Return ONLY the JSON string as specified in the instructions. Do not include any additional text, explanations, or justifications.
Let's start to analyze the video following the steps outlined above step by step.
"""

In [114]:
text_prompt = """
Objective:

Identify a 5-second time range within the provided video clip that best meets the following conditions, and provide a score for each condition, as well as a total score. The selected range must feature a **single individual**. The face must belong to the **same person** throughout the entire 5-second duration. Any switch to a different person's face is unacceptable. The person's facial features (e.g., hair, skin tone, face shape) must remain **consistent** throughout the selected range. The person's face should maintain a relatively **consistent angle** towards the camera. Significant changes in head orientation are not acceptable.

Conditions:

Speaking: The person must be speaking clearly and audibly.
Immobility: The person must be either sitting or standing completely still. Any visible movement, including but not limited to head nods, shakes, fidgeting, or shifting weight, is unacceptable. The person's posture should remain virtually unchanged throughout the selected range.
Unobstructed Mouth: When the person is speaking, their mouth must be 100% unobstructed. No part of any object (including hands, clothing, microphones, or text overlays) can be in front of the mouth at any time during speech. The mouth must be clearly and fully visible for the entire 5-second duration.
Appropriate Face Angle: The person's face should be angled towards the camera, allowing for a clear view of their features.
Good Face Lighting: The person's face should be well-lit and easily visible.
Clean Background: The background should be relatively uncluttered and free of distractions.
Consistent Lighting: The lighting must be stable, with no noticeable flickering or abrupt changes in brightness. Gradual changes in background lighting (e.g., clouds passing in front of the sun) are acceptable, but any sudden or significant change in the overall brightness of the person's face is unacceptable.
Appropriate Person Size: The person's face should occupy a reasonable portion of the frame, neither too small nor too large.

Scoring System (for each 5-second range):

For each condition, assign a score from 0 to 10, where:

10 = Perfectly meets the condition.
5 = Partially meets the condition.
0 = Does not meet the condition at all.

Selection Process Steps:

1.Review the entire video clip.
2.Identify all potential 5-second time ranges where the person is speaking.
3.Mitigating the Risk of Missing Mouth Obstructions: To ensure the "Unobstructed Mouth" condition is strictly met, use the following techniques:
- Review the video frame-by-frame: This is the most accurate method, but also the most time-consuming.
- Slow down the playback speed: This makes it easier to catch quick movements.
- Utilize video analysis software (if available): Some software can automatically detect objects (like hands) and track their movement, making it easier to identify potential obstructions.
- Consider multiple reviewers: Different people may notice different things.
4.For each identified time range, calculate a total score by summing the scores for Face Angle, Background Cleanliness, Person Size, Person Movement, Face Lighting, and Unobstructed Mouth. Before scoring, thoroughly verify each potential range, frame-by-frame, to ensure it absolutely meets all conditions. Discard any range that fails to meet even a single condition. **Ensure that the face belongs to the same person throughout the entire 5-second duration.**
5.Select the 5-second time range with the highest total score. If no range meets all conditions, indicate "No suitable range found."

Your response should be a JSON string with the following structure:
{
  "start_time": [Start Time in seconds],
  "end_time": [End Time in seconds],
  "face_angle_score": [Score],
  "background_cleanliness_score": [Score],
  "person_size_score": [Score],
  "person_movement_score": [Score],
  "face_lighting_score": [Score],
  "unobstructed_mouth_score": [Score],
  "total_score": [Total Score],
  "suitable_range_found": [true/false]
}

Example Output (with a suitable range found):
{
  "start_time": 15,
  "end_time": 20,
  "face_angle_score": 8,
  "background_cleanliness_score": 8,
  "person_size_score": 7,
  "person_movement_score": 10,
  "face_lighting_score": 8,
  "unobstructed_mouth_score": 10,
  "total_score": 51,
  "suitable_range_found": true
}

Example Output (if no suitable range is found):
{
  "start_time": null,
  "end_time": null,
  "face_angle_score": null,
  "background_cleanliness_score": null,
  "person_size_score": null,
  "person_movement_score": null,
  "face_lighting_score": null,
  "unobstructed_mouth_score": null,
  "total_score": null,
  "suitable_range_found": false
}

Return the JSON string as specified in the instructions and include any additional text, explanations, or justifications.
Let's start to analyze the video following the Selection Process Steps above step by step.
"""

In [24]:
text_prompt = """
Objective:

Identify a 5-second time range within the provided video clip that best meets the following conditions, and provide a score for each condition, as well as a total score. The selected range must feature a **single individual**. The face must belong to the **same person** throughout the entire 5-second duration. Any switch to a different person's face is unacceptable. The person's facial features (e.g., hair, skin tone, face shape) must remain **consistent** throughout the selected range. The person's face should maintain a relatively **consistent angle** towards the camera. Significant changes in head orientation are not acceptable.

Conditions:

Speaking: The person must be speaking clearly and audibly.
Immobility: The person must be either sitting or standing completely still. Any visible movement, including but not limited to head nods, shakes, fidgeting, or shifting weight, is unacceptable. The person's posture should remain virtually unchanged throughout the selected range.
Unobstructed Mouth: When the person is speaking, their mouth must be 100% unobstructed. No part of any object (including hands, clothing, microphones, or text overlays) can be in front of the mouth at any time during speech. The mouth must be clearly and fully visible for the entire 5-second duration.
Appropriate Face Angle: The person's face should be angled towards the camera, allowing for a clear view of their features.
Good Face Lighting: The person's face should be well-lit and easily visible.
Clean Background: The background should be relatively uncluttered and free of distractions.
Consistent Lighting: The lighting must be stable, with no noticeable flickering or abrupt changes in brightness. Gradual changes in background lighting (e.g., clouds passing in front of the sun) are acceptable, but any sudden or significant change in the overall brightness of the person's face is unacceptable.
Appropriate Person Size: The person's face should occupy a reasonable portion of the frame, neither too small nor too large.

Scoring System (for each 5-second range):

For each condition, assign a score from 0 to 10, where:

10 = Perfectly meets the condition.
5 = Partially meets the condition.
0 = Does not meet the condition at all.

Selection Process Steps:

1. Review the entire video clip.
2. **Randomly determine the order in which you will evaluate the following criteria: Face Angle, Background Cleanliness, Person Size, Person Movement, Face Lighting, and Unobstructed Mouth.** This randomization is crucial to minimize bias.
3. **Mitigating the Risk of Missing Mouth Obstructions:** To ensure the "Unobstructed Mouth" condition is strictly met, use the following techniques:
    - Review the video frame-by-frame: This is the most accurate method, but also the most time-consuming.
    - Slow down the playback speed: This makes it easier to catch quick movements.
    - Utilize video analysis software (if available): Some software can automatically detect objects (like hands) and track their movement, making it easier to identify potential obstructions.
    - Consider multiple reviewers: Different people may notice different things.
4. For each criterion, re-watch the video clip and focus specifically on that criterion.
5. Assign a score from 0 to 10 for each criterion.
6. Calculate a total score by summing the scores for all criteria.
7. Select the 5-second time range with the highest total score. If no range meets all conditions, indicate "No suitable range found."

Your response should be a JSON string with the following structure:
{
 "start_time": [Start Time in seconds],
 "end_time": [End Time in seconds],
 "face_angle_score": [Score],
 "background_cleanliness_score": [Score],
 "person_size_score": [Score],
 "person_movement_score": [Score],
 "face_lighting_score": [Score],
 "unobstructed_mouth_score": [Score],
 "total_score": [Total Score],
 "suitable_range_found": [true/false]
}

Example Output (with a suitable range found):
{
 "start_time": 15,
 "end_time": 20,
 "face_angle_score": 8,
 "background_cleanliness_score": 8,
 "person_size_score": 7,
 "person_movement_score": 10,
 "face_lighting_score": 8,
 "unobstructed_mouth_score": 10,
 "total_score": 51,
 "suitable_range_found": true
}

Example Output (if no suitable range is found):
{
 "start_time": null,
 "end_time": null,
 "face_angle_score": null,
 "background_cleanliness_score": null,
 "person_size_score": null,
 "person_movement_score": null,
 "face_lighting_score": null,
 "unobstructed_mouth_score": null,
 "total_score": null,
 "suitable_range_found": false
}

Return the JSON string as specified in the instructions and include any additional text, explanations, or justifications.
Let's start to analyze the video following the Selection Process Steps above step by step.
"""

In [45]:
# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [ ]:
class VisualDeepfakeDetector:
    def __init__(self, prompt, model="gemini-2.0-flash-001"): #settings
        self.model = model
        # generation config
        self.generate_content_config = types.GenerateContentConfig(
            temperature = 0.1,
            top_p = 0.95,
            max_output_tokens = 8192,
            response_modalities = ["TEXT"],
            safety_settings = [types.SafetySetting(
              category="HARM_CATEGORY_HATE_SPEECH",
              threshold="OFF"
            ),types.SafetySetting(
              category="HARM_CATEGORY_DANGEROUS_CONTENT",
              threshold="OFF"
            ),types.SafetySetting(
              category="HARM_CATEGORY_SEXUALLY_EXPLICIT",
              threshold="OFF"
            ),types.SafetySetting(
              category="HARM_CATEGORY_HARASSMENT",
              threshold="OFF"
            )],
        )
        self.client = genai.Client(
            vertexai=True,
            project="deepfake-422901",
            location="us-east1",
        )
    
        self.prompt = prompt
    
    def extract_json_text(self, text):
        """Extracts JSON from a text string using regular expressions."""
        try:
            json_match = re.search(r"\{.*\}", text, re.DOTALL)  # Find JSON block
            if json_match:
                return json_match.group(0)
            else:
                logger.warning("No JSON found in text.")
                return None
        except Exception as e:
            logger.exception("Error extracting JSON.")
            return None
    
    def generate(self, video_path, max_retries=3, initial_delay=1):
        """Generates content using the Gemini API with retries and logging."""
        retries = 0
        success_flag = False
        api_params = {  # Log API parameters
            "model": self.model,
            "temperature": self.generate_content_config.temperature,
            "top_p": self.generate_content_config.top_p,
            "max_output_tokens": self.generate_content_config.max_output_tokens,
            "safety_settings": self.generate_content_config.safety_settings
        }

        try:
            with open(video_path, "rb") as f:
                video_data = f.read()
            video_part = types.Part.from_bytes(mime_type="video/mp4", data=video_data)
            contents = [
                types.Content(
                    role="user",
                    parts=[
                        types.Part.from_text(text=self.prompt),
                        video_part
                    ]
                )
            ]

            while retries <= max_retries and not success_flag:
                try:
                    logger.info(f"Attempting API call (retry {retries}/{max_retries}). API Parameters: {api_params}")
                    complete_text = ""
                    for chunk in self.client.models.generate_content_stream(
                        model=self.model,
                        contents=contents,
                        config=self.generate_content_config,
                    ):
                        complete_text += chunk.text

                    logger.debug(f"Raw API Response: {complete_text}")  # Log raw response

                    clean_text = self.extract_json_text(complete_text)
                    if not clean_text:
                        raise ValueError("Could not extract JSON from API response.")

                    logger.debug(f"Extracted JSON: {clean_text}")

                    try:
                        json_response = json.loads(clean_text)
                        success_flag = True
                        return json_response
                    except json.JSONDecodeError as e:
                        logger.exception("Error decoding JSON.")
                        raise  # Re-raise for retry logic

                except Exception as e:  # Catch specific API exceptions
                    retries += 1
                    if retries <= max_retries:
                        delay = initial_delay * (2 ** (retries - 1))  # Exponential backoff
                        logger.warning(f"An error occurred: {e}. Retrying in {delay} seconds...")
                        time.sleep(delay)
                    else:
                        logger.error(f"Maximum retries ({max_retries}) exceeded. Giving up.")
                        raise

        except FileNotFoundError:
            logger.error(f"Video file not found: {video_path}")
            raise
        except Exception as e:
            logger.exception("Unexpected error during API call.")
            raise  

    def run(self, video_path):
        # self.logger = utils.get_logger()
        # self.logger.module_logger("call_to_action_detector_start")
        try:
            res = self.generate(video_path)
        # self.logger.module_logger("call_to_action_detector_done")
            return res
        except Exception as e:
            logger.error(f"Error during run: {e}")
            return None

In [16]:
import os
import tqdm

In [18]:
detector = VisualDeepfakeDetector(prompt=text_prompt, model="gemini-2.0-flash-001")

In [ ]:
safe_video_target_time = {}

safe_video_path = "./data/datasets/DemoDataset/videos/safe_videos"
for video_type in tqdm.tqdm(os.listdir(safe_video_path)):
    if video_type in safe_vedio_exp_cat: # sample portion for exp
        print(video_type)
        video_type_path = os.path.join(safe_video_path, video_type)
        for video in tqdm.tqdm(os.listdir(video_type_path)):
            print('Start process:', video)
            video_path = os.path.join(video_type_path, video)
            res = detector.run(video_path)
            safe_video_target_time.setdefault('video', []).append(video)
            safe_video_target_time.setdefault('label', []).append('safe')
            safe_video_target_time.setdefault('start_time', []).append(res['start_time'])
            safe_video_target_time.setdefault('end_time', []).append(res['end_time'])

In [63]:
import pandas as pd

In [ ]:
safe_video_target_time_df = pd.DataFrame.from_dict(safe_video_target_time)
safe_video_target_time_df.head()

In [ ]:
print(safe_video_target_time_df[safe_video_target_time_df['start_time'].isna()].shape)
safe_video_target_time_df[safe_video_target_time_df['start_time'].isna()]

In [ ]:
result_by_safe_video = {}

for i in range(len(safe_video_target_time["video"])):
    video_name = safe_video_target_time["video"][i]
    result_by_safe_video[video_name] = {
        "label": safe_video_target_time["label"][i],
        "start_time": safe_video_target_time["start_time"][i],
        "end_time": safe_video_target_time["end_time"][i],
    }

with open("result_by_safe_video_exp.json", "w", encoding="utf-8") as f:
    json.dump(result_by_safe_video, f, indent=2, ensure_ascii=False)

In [ ]:
scam_video_target_time = {}

scam_video_path = "./data/datasets/DemoDataset/videos/scam_videos"
for video in tqdm.tqdm(os.listdir(scam_video_path)):
    print('Start process:', video)
    video_path = os.path.join(scam_video_path, video)
    res = detector.run(video_path)
    scam_video_target_time.setdefault('video', []).append(video)
    scam_video_target_time.setdefault('label', []).append('scam')
    scam_video_target_time.setdefault('start_time', []).append(res['start_time'])
    scam_video_target_time.setdefault('end_time', []).append(res['end_time'])

In [ ]:
scam_video_target_time_df = pd.DataFrame.from_dict(scam_video_target_time)
scam_video_target_time_df.head()

In [ ]:
scam_video_target_time_df[scam_video_target_time_df['start_time'].isna()]

In [ ]:
result_by_scam_video = {}

for i in range(len(scam_video_target_time["video"])):
    video_name = scam_video_target_time["video"][i]
    result_by_scam_video[video_name] = {
        "label": scam_video_target_time["label"][i],
        "start_time": scam_video_target_time["start_time"][i],
        "end_time": scam_video_target_time["end_time"][i],
    }

with open("result_by_scam_video_exp.json", "w", encoding="utf-8") as f:
    json.dump(result_by_scam_video, f, indent=2, ensure_ascii=False)